In [1]:
!pip install soundfile

  Obtaining dependency information for soundfile from https://files.pythonhosted.org/packages/14/e9/6b761de83277f2f02ded7e7ea6f07828ec78e4b229b80e4ca55dd205b9dc/soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.0 MB 991.0 kB/s eta 0:00:01
   -------- ------------------------------- 0.2/1.0 MB 2.5 MB/s eta 0:00:01
   ------------------------- -------------- 0.6/1.0 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 6.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [88]:
import os
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

## По пути в input_path должен лежать набор файлов с одним говорящим. Дополнительные даннные не требуются

In [89]:
input_path = r"C:\separataion_train\dataset\archive (1)\golos\golos\0"

In [90]:
import torch
import torchaudio
import random
import os

def rms(waveform: torch.Tensor) -> float:
    return torch.sqrt(torch.mean(waveform ** 2))

def resample_wav(wav, sr, target_sr=8000):
    if sr != target_sr:
        wav = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)(wav)
    return wav

def mix_and_save(
    path1: str,
    path2: str,
    out_source1: str,
    out_source2: str,
    out_mix: str,
    target_snr_db: float = None,
    max_offset_sec: float = 0.0,
) -> bool:
    """
    Загружает два монофайла, нормирует их RMS, опционально смещает второй относительно первого,
    задаёт target_snr_db, складывает, нормирует по пику и сохраняет mix, source1, source2.
    Возвращает False, если один из исходников слишком тихой.
    """
    w1, sr1 = torchaudio.load(path1)
    w2, sr2 = torchaudio.load(path2)
    if sr1 != sr2:
        raise ValueError(f"Sample rates differ: {sr1} vs {sr2}")
    # Сделаем оба моно (если стерео, возьмём первый канал)
    if w1.shape[0] > 1:
        w1 = w1[:1]
    if w2.shape[0] > 1:
        w2 = w2[:1]

    # Опциональный случайный сдвиг (offset) второго файла относительно первого
    if max_offset_sec and max_offset_sec > 0:
        max_offset_samples = int(max_offset_sec * sr1)
        offset = random.randint(-max_offset_samples, max_offset_samples)
    else:
        offset = 0
    if offset > 0:
        if w2.shape[1] <= offset or w1.shape[1] <= offset:
            return False
        w2 = w2[:, offset:]
        w1 = w1[:, : w2.shape[1]]
    elif offset < 0:
        off = -offset
        if w1.shape[1] <= off or w2.shape[1] <= off:
            return False
        w1 = w1[:, off:]
        w2 = w2[:, : w1.shape[1]]
    if offset == 0:
        min_len = min(w1.shape[1], w2.shape[1])
        w1 = w1[:, :min_len]
        w2 = w2[:, :min_len]
    # Проверка уровня
    rms1 = rms(w1)
    rms2 = rms(w2)
    if rms1 < 1e-8 or rms2 < 1e-8:
        return False
    # Нормировка до единицы RMS
    w1_norm = w1 / rms1
    w2_norm = w2 / rms2
    if target_snr_db is not None:
        scale2 = 10 ** (-target_snr_db / 20)
        w2_norm = w2_norm * scale2
    # Сложение
    mix = w1_norm + w2_norm
    peak = mix.abs().max()
    if peak > 1:
        mix = mix / peak
        w1_norm = w1_norm / peak
        w2_norm = w2_norm / peak

    target_sr = 8000
    mix = resample_wav(mix, sr1, target_sr)
    w1_norm = resample_wav(w1_norm, sr1, target_sr)
    w2_norm = resample_wav(w2_norm, sr1, target_sr)

    torchaudio.save(out_mix, mix, target_sr)
    torchaudio.save(out_source1, w1_norm, target_sr)
    torchaudio.save(out_source2, w2_norm, target_sr)
    return True

In [91]:
def make_dir(dir_path):
    try:
       os.makedirs(dir_path)
    except FileExistsError:
       pass
    

In [92]:
def create_dataset(file_list, output_path):
    output_source1_dir = os.path.join(output_path, "source1")
    output_source2_dir = os.path.join(output_path, "source2")
    output_mixture_dir = os.path.join(output_path, "mixture")
    make_dir(output_source1_dir)
    make_dir(output_source2_dir)
    make_dir(output_mixture_dir)
    for i in tqdm(range(0, len(file_list) - 1, 2)):
        file_name_1, file_name_2 = os.path.join(input_path, file_list[i]), os.path.join(input_path, file_list[i + 1])
        output_source1, output_source2, output_mixture = os.path.join(output_source1_dir, str(i // 2) + ".wav"),os.path.join(output_source2_dir, str(i // 2) + ".wav"), os.path.join(output_mixture_dir, str(i // 2) + ".wav")

        mix_and_save(file_name_1, file_name_2, output_source1, output_source2, output_mixture)

In [93]:
def get_recursive_file_list(input_path):
    if os.path.isfile(input_path):
        if ".wav" in input_path or ".flac" in input_path:
            return [input_path]
        return []
    answer = []
    for file_path in os.listdir(input_path):
        answer.extend(get_recursive_file_list(os.path.join(input_path, file_path)))
    return answer

### Загрузка файлов

In [1]:
file_list = os.listdir(input_path)
file_list = file_list[:5000]

NameError: name 'os' is not defined

In [95]:
file_list[:10]

['00000eb80a7d11bfdce412b5c7cefa42.opus',
 '00001d3fd5d2c8cd00882afe632a6b0c.opus',
 '00003a1eacc4ff6d4dbb9a7dd0815db3.opus',
 '000045182196db407d62e18d58b062b6.opus',
 '00007e149372deb9f8dde3c7c155823d.opus',
 '000084a4edfa31b8a687ddf64202e887.opus',
 '00009b10c8a951d6bd5f640ab00e4cee.opus',
 '0000a1366ebe3ff75bc5d8f6993fd830.opus',
 '0000b1f1f1831a47ffbc980015bc5d88.opus',
 '0000bb8e1f6142c484850fbeaf10739e.opus']

Разбиение выборки на тренеровочную, тестовую, валидационнную

In [96]:
train_data, test_data__ = train_test_split(file_list, test_size=0.1)

In [97]:
val_data, test_data = train_test_split(test_data__, test_size=0.5)

In [98]:
output_path = "C:\\separataion_train\\dataset\\"

In [99]:
create_dataset(train_data, os.path.join(output_path, "train"))

100%|███████████████████████████████████████████████████████████████████████████████████████████| 2250/2250 [00:31<00:00, 71.69it/s]


In [100]:
create_dataset(val_data, os.path.join(output_path, "valid"))

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 125/125 [00:01<00:00, 68.96it/s]


In [101]:
create_dataset(test_data, os.path.join(output_path, "test"))

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 125/125 [00:01<00:00, 73.81it/s]


## Код скопированный из рапозитория speechbrain

In [102]:
"""
The .csv preparation functions for WSJ0-Mix.

Author
 * Cem Subakan 2020

 """

import csv
import os


def prepare_wsjmix(
    datapath,
    savepath,
    n_spks=2,
    skip_prep=False,
    librimix_addnoise=False,
    fs=8000,
):
    """
    Prepared wsj2mix if n_spks=2 and wsj3mix if n_spks=3.

    Arguments:
    ----------
        datapath (str) : path for the wsj0-mix dataset.
        savepath (str) : path where we save the csv file.
        n_spks (int): number of speakers
        skip_prep (bool): If True, skip data preparation
        librimix_addnoise: If True, add whamnoise to librimix datasets
    """

    if skip_prep:
        return

    if "wsj" in datapath:
        if n_spks == 2:
            assert (
                "2speakers" in datapath
            ), "Inconsistent number of speakers and datapath"
            create_wsj_csv(datapath, savepath)
        elif n_spks == 3:
            assert (
                "3speakers" in datapath
            ), "Inconsistent number of speakers and datapath"
            create_wsj_csv_3spks(datapath, savepath)
        else:
            raise ValueError("Unsupported Number of Speakers")
    else:
        print("Creating a csv file for a custom dataset")
        create_custom_dataset(datapath, savepath)


def create_custom_dataset(
    datapath,
    savepath,
    dataset_name="custom",
    set_types=["train", "valid", "test"],
    folder_names={
        "source1": "source1",
        "source2": "source2",
        "mixture": "mixture",
    },
):
    """
    This function creates the csv file for a custom source separation dataset
    """

    for set_type in set_types:
        mix_path = os.path.join(datapath, set_type, folder_names["mixture"])
        s1_path = os.path.join(datapath, set_type, folder_names["source1"])
        s2_path = os.path.join(datapath, set_type, folder_names["source2"])

        files = os.listdir(mix_path)

        mix_fl_paths = [os.path.join(mix_path, fl) for fl in files]
        s1_fl_paths = [os.path.join(s1_path, fl) for fl in files]
        s2_fl_paths = [os.path.join(s2_path, fl) for fl in files]

        csv_columns = [
            "ID",
            "duration",
            "mix_wav",
            "mix_wav_format",
            "mix_wav_opts",
            "s1_wav",
            "s1_wav_format",
            "s1_wav_opts",
            "s2_wav",
            "s2_wav_format",
            "s2_wav_opts",
            "noise_wav",
            "noise_wav_format",
            "noise_wav_opts",
        ]

        with open(
            os.path.join(savepath, dataset_name + "_" + set_type + ".csv"),
            "w",
            encoding="utf-8",
        ) as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
            writer.writeheader()
            for i, (mix_path, s1_path, s2_path) in enumerate(
                zip(mix_fl_paths, s1_fl_paths, s2_fl_paths)
            ):
                row = {
                    "ID": i,
                    "duration": 1.0,
                    "mix_wav": mix_path,
                    "mix_wav_format": "wav",
                    "mix_wav_opts": None,
                    "s1_wav": s1_path,
                    "s1_wav_format": "wav",
                    "s1_wav_opts": None,
                    "s2_wav": s2_path,
                    "s2_wav_format": "wav",
                    "s2_wav_opts": None,
                }
                writer.writerow(row)


def create_wsj_csv(datapath, savepath):
    """
    This function creates the csv files to get the speechbrain data loaders for the wsj0-2mix dataset.

    Arguments:
        datapath (str) : path for the wsj0-mix dataset.
        savepath (str) : path where we save the csv file
    """
    for set_type in ["tr", "cv", "tt"]:
        mix_path = os.path.join(datapath, "wav8k/min/" + set_type + "/mix/")
        s1_path = os.path.join(datapath, "wav8k/min/" + set_type + "/s1/")
        s2_path = os.path.join(datapath, "wav8k/min/" + set_type + "/s2/")

        files = os.listdir(mix_path)

        mix_fl_paths = [mix_path + fl for fl in files]
        s1_fl_paths = [s1_path + fl for fl in files]
        s2_fl_paths = [s2_path + fl for fl in files]

        csv_columns = [
            "ID",
            "duration",
            "mix_wav",
            "mix_wav_format",
            "mix_wav_opts",
            "s1_wav",
            "s1_wav_format",
            "s1_wav_opts",
            "s2_wav",
            "s2_wav_format",
            "s2_wav_opts",
        ]

        with open(
            savepath + "/wsj_" + set_type + ".csv",
            "w",
            newline="",
            encoding="utf-8",
        ) as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
            writer.writeheader()
            for i, (mix_path, s1_path, s2_path) in enumerate(
                zip(mix_fl_paths, s1_fl_paths, s2_fl_paths)
            ):
                row = {
                    "ID": i,
                    "duration": 1.0,
                    "mix_wav": mix_path,
                    "mix_wav_format": "wav",
                    "mix_wav_opts": None,
                    "s1_wav": s1_path,
                    "s1_wav_format": "wav",
                    "s1_wav_opts": None,
                    "s2_wav": s2_path,
                    "s2_wav_format": "wav",
                    "s2_wav_opts": None,
                }
                writer.writerow(row)


def create_wsj_csv_3spks(datapath, savepath):
    """
    This function creates the csv files to get the speechbrain data loaders for the wsj0-3mix dataset.

    Arguments:
        datapath (str) : path for the wsj0-mix dataset.
        savepath (str) : path where we save the csv file
    """
    for set_type in ["tr", "cv", "tt"]:
        mix_path = os.path.join(datapath, "wav8k/min/" + set_type + "/mix/")
        s1_path = os.path.join(datapath, "wav8k/min/" + set_type + "/s1/")
        s2_path = os.path.join(datapath, "wav8k/min/" + set_type + "/s2/")
        s3_path = os.path.join(datapath, "wav8k/min/" + set_type + "/s3/")

        files = os.listdir(mix_path)

        mix_fl_paths = [mix_path + fl for fl in files]
        s1_fl_paths = [s1_path + fl for fl in files]
        s2_fl_paths = [s2_path + fl for fl in files]
        s3_fl_paths = [s3_path + fl for fl in files]

        csv_columns = [
            "ID",
            "duration",
            "mix_wav",
            "mix_wav_format",
            "mix_wav_opts",
            "s1_wav",
            "s1_wav_format",
            "s1_wav_opts",
            "s2_wav",
            "s2_wav_format",
            "s2_wav_opts",
            "s3_wav",
            "s3_wav_format",
            "s3_wav_opts",
        ]

        with open(
            savepath + "/wsj_" + set_type + ".csv",
            "w",
            newline="",
            encoding="utf-8",
        ) as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
            writer.writeheader()
            for i, (mix_path, s1_path, s2_path, s3_path) in enumerate(
                zip(mix_fl_paths, s1_fl_paths, s2_fl_paths, s3_fl_paths)
            ):
                row = {
                    "ID": i,
                    "duration": 1.0,
                    "mix_wav": mix_path,
                    "mix_wav_format": "wav",
                    "mix_wav_opts": None,
                    "s1_wav": s1_path,
                    "s1_wav_format": "wav",
                    "s1_wav_opts": None,
                    "s2_wav": s2_path,
                    "s2_wav_format": "wav",
                    "s2_wav_opts": None,
                    "s3_wav": s3_path,
                    "s3_wav_format": "wav",
                    "s3_wav_opts": None,
                }
                writer.writerow(row)

## Подготовка финальных .csv файлов

In [103]:
create_custom_dataset(output_path, output_path)